# 06 · TabNet (정형 전용 딥러닝, 내장 해석력)

> 역할: 정형 데이터에 특화된 딥러닝 TabNet으로, 트리/MLP/FT-Transformer와 비교한다.
> 특징: 매 단계마다 '어떤 피처를 볼지' 스스로 골라서, 트리처럼 **내장 피처 중요도**를 준다.
> -> 03의 LightGBM SHAP와 '어떤 변수가 중요한가'를 맞비교할 수 있다.

## 0. 환경 설정
- **pytorch-tabnet**: TabNet(구글, 2019)을 sklearn처럼 쉽게 쓰게 해주는 라이브러리.
  TabNet은 'sequential attention'으로 결정 단계마다 일부 피처만 골라 추론하는 정형 전용 신경망이다.
  분류·회귀에 쓰며, fit/predict_proba/feature_importances_ 등 익숙한 인터페이스를 제공한다.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pytorch_tabnet.tab_model import TabNetClassifier

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR     = PROJECT_ROOT / "processed"
RESULTS_CSV = PROJECT_ROOT / "notebooks" / "results.csv"
print("준비 완료")

## 1. 데이터 불러오기
TabNet은 numpy 배열을 받는다. X는 float32, y는 정수(int)로 준비한다.

In [ ]:
train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]

X_train = train_df[feature_cols].values.astype("float32")
y_train = train_df[TARGET].values.astype("int64")
X_val   = val_df[feature_cols].values.astype("float32")
y_val   = val_df[TARGET].values.astype("int64")

print("피처 수:", len(feature_cols), " train 양성:", int(y_train.sum()))

## 2. 학습용 샘플 (CPU 대응)
TabNet도 CPU에선 무겁다. 05와 같은 방식으로 **학습은 일부만(비율 유지), 평가는 val 전체**.
느리면 SUBSAMPLE이나 max_epochs를 더 줄이면 된다.

In [ ]:
SUBSAMPLE = 100000
rng = np.random.default_rng(42)
sel = rng.choice(len(X_train), size=min(SUBSAMPLE, len(X_train)), replace=False)
X_fit, y_fit = X_train[sel], y_train[sel]
print("학습 샘플:", len(sel), " 양성:", int(y_fit.sum()))

## 3. TabNet 학습 + 불균형 처리
TabNet은 매 결정 단계에서 sequential attention으로 **볼 피처를 스스로 선택**한다(미분 가능한 피처 선택).
그래서 신경망이면서도 트리처럼 어떤 피처가 중요한지 알려준다.
- `weights=1`: TabNet 내장 클래스 균형 -> 소수 클래스(Yes)를 자동으로 더 비중 있게 학습(불균형 대응).
- `eval_set` + `patience`: val 성능이 더 안 오르면 일찍 멈춘다(early stopping).

In [ ]:
set_seed(42)
clf = TabNetClassifier(seed=42, verbose=1)
clf.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    eval_name=["val"],
    eval_metric=["auc"],
    max_epochs=15,
    patience=5,
    batch_size=16384,
    virtual_batch_size=1024,
    weights=1,
)

## 4. 평가 + 기록
val 전체로 평가해 같은 지표로 results.csv에 기록.

In [ ]:
val_prob = clf.predict_proba(X_val)[:, 1]
metrics = compute_metrics(y_val, val_prob)
log_result("TabNet", metrics, path=str(RESULTS_CSV))
print("TabNet", metrics)

## 5. TabNet 내장 피처 중요도
TabNet이 '어떤 변수를 중요하게 봤는지' 막대그래프로. 03의 LightGBM SHAP와 비교해보면
두 방식(신경망 vs 트리)이 같은 변수를 중요하다고 보는지 확인할 수 있다(보고서에 좋은 비교).

In [ ]:
imp = clf.feature_importances_
order = np.argsort(imp)[::-1]
names = [feature_cols[i] for i in order]

plt.figure(figsize=(8, 7))
plt.barh(names[::-1], imp[order][::-1])
plt.title("TabNet feature importance")
plt.tight_layout()
plt.show()

## 6. 결과 비교표
지금까지 모든 모델 비교 (PR_AUC 내림차순).

In [ ]:
res = pd.read_csv(RESULTS_CSV)
res = res.drop_duplicates(subset="model", keep="last")
res = res.sort_values("PR_AUC", ascending=False)
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

## 7. 저장 (분석용 결과 파일)
val 확률을 저장해 08_Decision_Analysis가 쓰게 한다.

In [ ]:
np.save(PROJECT_ROOT / "notebooks" / "tabnet_val_prob.npy", val_prob)
print("저장 완료: tabnet_val_prob.npy")

---
### 정리
- TabNet: 정형 전용 딥러닝. sequential attention으로 피처를 단계적으로 선택, 내장 피처 중요도 제공.
- 비교 포인트: PR_AUC가 트리/MLP/FT-Transformer 대비 어디인가.
- 해석 비교: TabNet 피처 중요도 vs 03의 LightGBM SHAP — 두 방식이 같은 변수를 중요하다고 보는가?
- 비용: TabNet도 CPU에선 트리보다 훨씬 무겁다(05와 같은 교훈).

다음: 07_AutoEncoder(이상탐지 관점), 08_Decision_Analysis(통합 + 비용 threshold).